In [1]:
import numpy as np
from scipy.stats import norm

def calculate_portfolio_var():
    # ==========================================
    # 1. 输入参数 (Inputs)
    # ==========================================
    # 资产 A
    shares_a = -10000.0
    price_a = 50.0
    sigma_a = 0.35      # 波动率
    beta_a = 0.25

    # 资产 B
    shares_b = 10000.0
    price_b = 100.0
    sigma_b = 0.45
    beta_b = 1.50

    # 市场参数
    sigma_m = 0.30      # 市场波动率
    rf = 0.04           # 无风险利率
    rm_expected = 0.11  # 市场预期回报率
    
    # VaR 参数
    confidence_level = 0.95
    annual_factor = 250

    # ==========================================
    # 2. 投资组合基础计算 (Portfolio Basics)
    # ==========================================
    val_a = shares_a * price_a
    val_b = shares_b * price_b
    val_p = val_a + val_b  # 总价值
    
    w_a = val_a / val_p    # A 的权重
    w_b = val_b / val_p    # B 的权重
    
    # 组合 Beta
    beta_p = w_a * beta_a + w_b * beta_b
    
    print("-" * 30)
    print(f"组合总价值 (V_p): ${val_p:,.2f}")
    print(f"组合 Beta: {beta_p:.4f}")
    print("-" * 30)

    # ==========================================
    # 3. 波动率计算 (Volatility Calculation)
    # ==========================================
    # 使用单指数模型假设 (Single Index Model) 计算协方差
    # Cov(A, B) = Beta_A * Beta_B * Var(Rm)
    cov_ab = beta_a * beta_b * (sigma_m ** 2)
    
    # 组合年化方差
    var_p_annual = (w_a**2 * sigma_a**2) + \
                   (w_b**2 * sigma_b**2) + \
                   (2 * w_a * w_b * cov_ab)
    
    sigma_p_annual = np.sqrt(var_p_annual)
    
    # 转换为日波动率
    sigma_p_daily = sigma_p_annual / np.sqrt(annual_factor)

    # ==========================================
    # 4. VaR 计算：对冲前 (Before Hedge)
    # ==========================================
    # 临界值 Z-score (95% 单尾)
    z_score = norm.ppf(confidence_level)
    
    # 预期日回报 (使用 CAPM)
    # E[Rp] = Rf + Beta_p * (Rm - Rf)
    mu_p_annual = rf + beta_p * (rm_expected - rf)
    mu_p_daily = mu_p_annual / annual_factor
    
    # VaR 公式: V * (Z * sigma - mu)
    # 注意：通常 VaR 表示为损失的正数，所以我们减去预期收益
    var_before = val_p * (z_score * sigma_p_daily - mu_p_daily)

    print(f"【对冲前】")
    print(f"年化方差: {var_p_annual:.4f}")
    print(f"日波动率: {sigma_p_daily:.4f}")
    print(f"日预期回报: {mu_p_daily:.6f}")
    print(f"95% 1-Day VaR: ${var_before:.4f}")
    print("-" * 30)

    # ==========================================
    # 5. VaR 计算：对冲后 (After Hedge)
    # ==========================================
    # 对冲策略：将 Beta 降为 0。这意味着移除了所有系统性风险。
    # 总方差 = 系统性方差 + 特质方差 (Specific Variance)
    
    # 系统性方差 (Systematic Variance) = (Beta_p^2) * (Sigma_m^2)
    var_systematic_annual = (beta_p ** 2) * (sigma_m ** 2)
    
    # 特质方差 (Specific Variance) = 总方差 - 系统性方差
    var_specific_annual = var_p_annual - var_systematic_annual
    
    # 对冲后的风险仅剩特质风险
    sigma_hedged_daily = np.sqrt(var_specific_annual) / np.sqrt(annual_factor)
    
    # 对冲后的预期回报
    # Beta = 0, 因此预期回报理论上回归到无风险利率 Rf (alpha 假设为 0)
    mu_hedged_daily = rf / annual_factor
    
    var_after = val_p * (z_score * sigma_hedged_daily - mu_hedged_daily)

    print(f"【对冲后 (Beta=0)】")
    print(f"系统性方差 (已移除): {var_systematic_annual:.4f}")
    print(f"特质方差 (保留): {var_specific_annual:.4f}")
    print(f"对冲后日波动率: {sigma_hedged_daily:.4f}")
    print(f"对冲后日预期回报 (Rf): {mu_hedged_daily:.6f}")
    print(f"95% 1-Day VaR: ${var_after:.4f}")
    print("-" * 30)

if __name__ == "__main__":
    calculate_portfolio_var()

------------------------------
组合总价值 (V_p): $500,000.00
组合 Beta: 2.7500
------------------------------
【对冲前】
年化方差: 0.7975
日波动率: 0.0565
日预期回报: 0.000930
95% 1-Day VaR: $45985.7363
------------------------------
【对冲后 (Beta=0)】
系统性方差 (已移除): 0.6806
特质方差 (保留): 0.1169
对冲后日波动率: 0.0216
对冲后日预期回报 (Rf): 0.000160
95% 1-Day VaR: $17702.3056
------------------------------
